# 03 · DPO Fine-Tuning

Fine-tunes Llama-3-8B-Instruct with DPO + LoRA (ART-003/004).
**Run on Google Colab with A100 GPU.**

Expected time: ~8–12 hours. Checkpointing is on — safe to resume if session times out.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import subprocess
subprocess.run(['git', 'clone', 'https://github.com/YOUR_USERNAME/AlignR', '/content/AlignR'])
%cd /content/AlignR
!pip install -r requirements.txt --quiet

In [ ]:
# Copy dataset and reward model from Drive
!cp -r /content/drive/MyDrive/alignr/pref_dataset /content/AlignR/data/pref_dataset
!cp -r /content/drive/MyDrive/alignr/reward_model /content/AlignR/reward_model

In [ ]:
# Run DPO fine-tuning
!python train/train_dpo.py \
    --config configs/dpo_default.yaml \
    --dataset data/pref_dataset/ \
    --reward-model ./reward_model/ \
    --output ./aligned_model_lora/ \
    --mlflow-uri http://localhost:5000

# If Colab times out, just re-run the cell above — training resumes from checkpoint.

In [ ]:
# Merge LoRA into base model (runs on CPU ~5 min)
!python train/merge_lora.py \
    --base meta-llama/Llama-3-8B-Instruct \
    --lora ./aligned_model_lora/ \
    --output ./aligned_model/

In [ ]:
# Save everything to Drive
!cp -r /content/AlignR/aligned_model_lora /content/drive/MyDrive/alignr/aligned_model_lora
!cp -r /content/AlignR/aligned_model /content/drive/MyDrive/alignr/aligned_model
print('Saved to Drive.')